# Lista SEL — Sistemas de Equações Lineares

In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

def fwd(L, b):
    n = len(b); y = np.zeros(n)
    for i in range(n): y[i] = (b[i] - L[i,:i]@y[:i]) / L[i,i]
    return y

def bck(U, b):
    n = len(b); x = np.zeros(n)
    for i in range(n-1,-1,-1): x[i] = (b[i] - U[i,i+1:]@x[i+1:]) / U[i,i]
    return x


## Questão 1
Gauss, LU e Cholesky para o circuito resistivo.

In [2]:
A = np.array([[8,-4,-2],[-4,6,-2],[-2,-2,10]], dtype=float)
b = np.array([10, 0, 4], dtype=float)
n = 3

# Eliminação de Gauss com pivotamento parcial
Ag, bg = A.copy(), b.copy()
for k in range(n-1):
    p = k + np.argmax(np.abs(Ag[k:, k]))
    Ag[[k,p]] = Ag[[p,k]]; bg[[k,p]] = bg[[p,k]]
    for i in range(k+1, n):
        m = Ag[i,k]/Ag[k,k]; Ag[i,k:] -= m*Ag[k,k:]; bg[i] -= m*bg[k]
x_gauss = bck(Ag, bg)
print("Gauss:", x_gauss)

# LU Doolittle com pivotamento
Ul, Pl, Ll = A.copy(), np.eye(n), np.eye(n)
for k in range(n):
    p = k + np.argmax(np.abs(Ul[k:,k]))
    Ul[[k,p]] = Ul[[p,k]]; Pl[[k,p]] = Pl[[p,k]]
    if k > 0: Ll[[k,p],:k] = Ll[[p,k],:k]
    for i in range(k+1,n):
        Ll[i,k] = Ul[i,k]/Ul[k,k]; Ul[i,k:] -= Ll[i,k]*Ul[k,k:]
x_lu = bck(Ul, fwd(Ll, Pl@b))
print("LU:   ", x_lu)

# Cholesky
try:
    Ch = np.linalg.cholesky(A); print("Cholesky L =\n", Ch)
except: print("Cholesky não aplicável")


Gauss: [2.758621 2.310345 1.413793]
LU:    [2.758621 2.310345 1.413793]
Cholesky L =
 [[ 2.828427  0.        0.      ]
 [-1.414214  2.        0.      ]
 [-0.707107 -1.5       2.692582]]


## Questão 2
Inversa da matriz e cálculo de Ix.

In [3]:
A = np.array([[14,4,4],[4,7,19],[4,7,18]], dtype=float)
b = np.array([100, 100, 100], dtype=float)
Ainv = np.linalg.inv(A)
I1, I2, Ix = Ainv @ b
print("A⁻¹ =\n", Ainv)
print(f"[I1, I2, Ix] = {Ainv@b}")
print(f"Ix = {Ix:.6f}")


A⁻¹ =
 [[ 0.085366  0.536585 -0.585366]
 [-0.04878  -2.878049  3.04878 ]
 [-0.        1.       -1.      ]]
[I1, I2, Ix] = [ 3.658537 12.195122  0.      ]
Ix = 0.000000


## Questão 3
Tensões nodais — Gauss 4×4.

In [4]:
A = np.array([[-4,2,0,1],[1,-2,1,0],[0,2,-3,1],[5,0,5,-12]], dtype=float)
b = np.array([-100, 0, 0, 0], dtype=float)
n = 4
Ag, bg = A.copy(), b.copy()
for k in range(n-1):
    p = k + np.argmax(np.abs(Ag[k:, k]))
    Ag[[k,p]] = Ag[[p,k]]; bg[[k,p]] = bg[[p,k]]
    for i in range(k+1, n):
        m = Ag[i,k]/Ag[k,k]; Ag[i,k:] -= m*Ag[k,k:]; bg[i] -= m*bg[k]
print("V =", bck(Ag, bg))


V = [76. 72. 68. 60.]


## Questão 4
Gauss-Seidel — verificação de convergência (tol = 10⁻²).

In [5]:
A = np.array([[20,-10,-4],[-10,20,-5],[-4,-5,20]], dtype=float)
b = np.array([26, 0, 7], dtype=float)
n = 3

dd = all(abs(A[i,i]) > sum(abs(A[i,j]) for j in range(n) if j!=i) for i in range(n))
print("Diagonal dominante estrita:", dd)

x, tol = np.zeros(n), 1e-2
for it in range(10000):
    xo = x.copy()
    for i in range(n): x[i] = (b[i] - A[i,:i]@x[:i] - A[i,i+1:]@xo[i+1:]) / A[i,i]
    if np.max(np.abs(x-xo)) / max(np.max(np.abs(x)),1) < tol:
        print(f"Gauss-Seidel ({it+1} iter): x = {x}")
        print(f"Erro relativo real = {np.max(np.abs(x - np.linalg.solve(A,b)))/np.max(np.abs(np.linalg.solve(A,b))):.2e}")
        break


Diagonal dominante estrita: True
Gauss-Seidel (6 iter): x = [2.214186 1.390227 1.140394]
Erro relativo real = 7.15e-03


## Questão 5
LU com pivotamento parcial.

In [6]:
A = np.array([[2,-6,-1],[-3,-1,7],[-8,1,-2]], dtype=float)
b = np.array([-38, -34, -20], dtype=float)
n = 3
Ul, Pl, Ll = A.copy(), np.eye(n), np.eye(n)
for k in range(n):
    p = k + np.argmax(np.abs(Ul[k:,k]))
    Ul[[k,p]] = Ul[[p,k]]; Pl[[k,p]] = Pl[[p,k]]
    if k > 0: Ll[[k,p],:k] = Ll[[p,k],:k]
    for i in range(k+1,n):
        Ll[i,k] = Ul[i,k]/Ul[k,k]; Ul[i,k:] -= Ll[i,k]*Ul[k,k:]
print("P =\n", Pl); print("L =\n", Ll); print("U =\n", Ul)
print("x =", bck(Ul, fwd(Ll, Pl@b)))


P =
 [[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]
L =
 [[ 1.       0.       0.     ]
 [-0.25     1.       0.     ]
 [ 0.375    0.23913  1.     ]]
U =
 [[-8.        1.       -2.      ]
 [ 0.       -5.75     -1.5     ]
 [ 0.        0.        8.108696]]
x = [ 4.  8. -2.]


## Questão 6
Decomposição de Crout.

In [7]:
A = np.array([[2,-6,1],[-1,7,-1],[1,-3,2]], dtype=float)
b = np.array([12, -8, 16], dtype=float)
n = 3
Lc, Uc = np.zeros((n,n)), np.eye(n)
for j in range(n):
    for i in range(j,n): Lc[i,j] = A[i,j] - sum(Lc[i,k]*Uc[k,j] for k in range(j))
    for i in range(j+1,n): Uc[j,i] = (A[j,i] - sum(Lc[j,k]*Uc[k,i] for k in range(j))) / Lc[j,j]
print("L =\n", Lc); print("U =\n", Uc)
print("Verificação L@U =\n", Lc@Uc)
print("x =", bck(Uc, fwd(Lc, b)))


L =
 [[ 2.   0.   0. ]
 [-1.   4.   0. ]
 [ 1.   0.   1.5]]
U =
 [[ 1.    -3.     0.5  ]
 [ 0.     1.    -0.125]
 [ 0.     0.     1.   ]]
Verificação L@U =
 [[ 2. -6.  1.]
 [-1.  7. -1.]
 [ 1. -3.  2.]]
x = [3.666667 0.333333 6.666667]


## Questão 7
Jacobi, Gauss-Seidel, SOR (ω=0.9 e ω=1.2) — reatores.

In [8]:
A = np.array([[15,-3,-1],[-3,18,-6],[-4,-1,12]], dtype=float)
b = np.array([3800, 1200, 2350], dtype=float)
n = 3

def iterar(metodo, w=1.0, tol=1e-6, maxit=10000):
    x = np.zeros(n)
    for it in range(maxit):
        xo = x.copy()
        if metodo == 'jacobi':
            for i in range(n): x[i] = (b[i] - A[i,:i]@xo[:i] - A[i,i+1:]@xo[i+1:]) / A[i,i]
        else:
            for i in range(n):
                gs = (b[i] - A[i,:i]@x[:i] - A[i,i+1:]@xo[i+1:]) / A[i,i]
                x[i] = (1-w)*xo[i] + w*gs
        if np.max(np.abs(x-xo)) / max(np.max(np.abs(x)),1) < tol:
            return x, it+1
    return x, maxit

for nome, kw in [('Jacobi',{'metodo':'jacobi'}),('Gauss-Seidel',{'metodo':'gs'}),
                  ('SOR w=0.9',{'metodo':'gs','w':0.9}),('SOR w=1.2',{'metodo':'gs','w':1.2})]:
    x, it = iterar(**kw)
    print(f"{nome:<14}: x={np.round(x,4)}  iter={it}")


Jacobi        : x=[320.2071 227.2019 321.5024]  iter=15
Gauss-Seidel  : x=[320.2072 227.202  321.5026]  iter=10
SOR w=0.9     : x=[320.2071 227.202  321.5025]  iter=13
SOR w=1.2     : x=[320.2072 227.2021 321.5026]  iter=16


## Questão 8
Sistema 6×6 — LU direto + Gauss-Seidel + normas + condicionamento.

In [9]:
A = np.array([
    [ 8.0,-4.0, 0.0,-1.0, 0.0, 0.0],
    [-4.0,11.5,-2.5, 0.0,-5.0, 0.0],
    [ 0.0,-2.5, 4.5, 0.0, 0.0,-2.0],
    [-1.0, 0.0, 0.0, 3.0,-2.0, 0.0],
    [ 0.0,-5.0, 0.0,-2.0, 8.5,-1.5],
    [ 0.0, 0.0,-2.0, 0.0,-1.5, 8.0]], dtype=float)
b = np.array([20,-12,14,8,-30,0], dtype=float)
n = 6

# LU (direto via numpy para 6x6)
x_dir = np.linalg.solve(A, b)
print("Solução direta (LU):", x_dir)

# Gauss-Seidel
x, tol = np.zeros(n), 1e-6
for it in range(10000):
    xo = x.copy()
    for i in range(n): x[i] = (b[i] - A[i,:i]@x[:i] - A[i,i+1:]@xo[i+1:]) / A[i,i]
    if np.max(np.abs(x-xo)) / max(np.max(np.abs(x)),1) < tol: break
print(f"Gauss-Seidel ({it+1} iter): {x}")

erro   = x_dir - x
res    = b - A@x
print(f"||x||_inf    = {np.max(np.abs(x_dir)):.6f}")
print(f"||erro||_inf = {np.max(np.abs(erro)):.3e}")
print(f"||res||_inf  = {np.max(np.abs(res)):.3e}")
print(f"cond(A)      = {np.linalg.cond(A):.6f}")


Solução direta (LU): [ 1.046969 -2.757569  1.268915 -0.593972 -5.414442 -0.697979]
Gauss-Seidel (50 iter): [ 1.046983 -2.757552  1.268928 -0.593954 -5.414426 -0.697973]
||x||_inf    = 5.414442
||erro||_inf = 1.832e-05
||res||_inf  = 3.133e-05
cond(A)      = 21.587852
